# [Semantic Kernel with Function Calling on native functions (plug-ins)](https://learn.microsoft.com/en-us/semantic-kernel/get-started/quick-start-guide?pivots=programming-language-python#writing-your-first-console-app)

In just a few steps, we can build our first AI agent with Semantic Kernel in either Python, .NET, or Java. This guide will show how to...

- Install the necessary packages
- Ceate a back-and-forth conversation with an AI
- Give an AI agent the ability to run your code
- Watch the AI create plans on the fly

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv

load_dotenv("./../config/credentials_my.env")
print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Create an Azure chat completion object e.g. the `assistant` from SK library

In [2]:
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
chat_completion=AzureChatCompletion(service_id="default")
chat_completion

AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='default', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001FBDB0DDD90>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)

# Create a user message and add it to a blank history

In [3]:
# Create a blank history of the conversation
from semantic_kernel.contents.chat_history import ChatHistory
history = ChatHistory() # initially blank

# Add user input to the history
history.add_user_message("Tell me what is Azure in less than 10 words.")

history

ChatHistory(messages=[ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Tell me what is Azure in less than 10 words.', encoding=None)], encoding=None, finish_reason=None)])

# Define settings for the chat
## Important: note that `function_choice_behavior=None`

In [4]:
# Create default settings for the chat conversation

from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)

execution_settings = AzureChatPromptExecutionSettings()
execution_settings

AzureChatPromptExecutionSettings(service_id=None, extension_data={}, function_choice_behavior=None, ai_model_id=None, frequency_penalty=None, logit_bias=None, max_tokens=None, number_of_responses=None, presence_penalty=None, seed=None, stop=None, stream=False, temperature=None, top_p=None, user=None, store=None, metadata=None, response_format=None, function_call=None, functions=None, messages=None, function_call_behavior=None, parallel_tool_calls=True, tools=None, tool_choice=None, structured_json_response=False, stream_options=None, extra_body=None)

# Initialize the kernel

In [5]:
from semantic_kernel import Kernel
kernel = Kernel()

kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001FBD8ACB4D0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Put all together
- Chat Completion
- History (including the first user message)
- Settings
- Kernel

The `result` we get is a list of semantic_kernel.contents.chat_message_content.**ChatMessageContent** whose `content` field contains the message text.

In [6]:
result = await chat_completion.get_chat_message_contents(
    chat_history=history,
    settings = execution_settings,
    kernel=kernel
)

#  Print the results
print(result)

print(f"\nAssistant's response: {result[0].content}")

[ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-An3xgws9D5Xbl4cBJWmTntOc9AlfE', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Microsoft's cloud computing service for building, testing, deploying applications.", refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1736256664, model='gpt-4o-2024-05-13', object='chat.completion', service_tier=None, system_fingerprint='fp_04751d0b65', usage=CompletionUsage(completion_tokens=13, prompt_tokens=19, total_tokens=32, completion_tokens_details=None, prompt_tokens_details=None), prompt_filter_results=[{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'fi

# What does the kernel contain? No services, no plugins so far

In [7]:
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001FBD8ACB4D0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

In [8]:
# no added history too
history

ChatHistory(messages=[ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Tell me what is Azure in less than 10 words.', encoding=None)], encoding=None, finish_reason=None)])

# Add the Chat Completion as a service to the kernel
The Chat Completion service has id = 'default' as defined above

In [9]:
kernel.add_service(chat_completion)

kernel 

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'default': AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='default', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001FBDB0DDD90>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=19, completion_tokens=13, total_tokens=32)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001FBD8ACB4D0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Set the logging level for  semantic_kernel.kernel to DEBUG
One of the main benefits of using Semantic Kernel is that it supports enterprise-grade services.<br/>
In this sample, we add the logging service to the kernel to help debug the AI agent.

In [10]:
import logging

logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
    
logging.getLogger("kernel").setLevel(logging.DEBUG)

# Add a ***Native*** Plugin to the Kernel
now the kernel has both
- a service, that we can get with kernel.get_service()
- a plugin, that we can retrieve with kernel.get_plugin("Lights")

In [11]:
# First, we define the plugin through its class...

from typing import Annotated
from semantic_kernel.functions import kernel_function

class LightsPlugin:
    lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": True},
    ]

    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights

    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

In [12]:
# ...then, we add the plugin to the kernel, using a new plugin name

kernel.add_plugin(
    LightsPlugin(),
    plugin_name="Lights",
)

kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'default': AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='default', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001FBDB0DDD90>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=19, completion_tokens=13, total_tokens=32)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001FBD8ACB4D0>, plugins={'Lights': KernelPlugin(name='Lights', description=None, functions={'change_state': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='change_state', plugin_name='Lights', description='Changes the state of the light', parameters=[KernelParameterMetadata(name='id', description=None, default_value=None, type_='int', is_required=True, type_object=<class 'int'>, schema_data={'type': 'integer'}, include_in_function_choices=True), KernelParameterMetadata(name='is_on', description=None, default_value=None, type_='bool', is_required=True, type_

# Create a new User message that can leverage the added plug-in

In [13]:
# Create a blank history of the conversation
history = ChatHistory() # initially blank

# Add user input to the history
history.add_user_message("Toggle the status of my second light.")

history

ChatHistory(messages=[ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Toggle the status of my second light.', encoding=None)], encoding=None, finish_reason=None)])

# Enable planning with Function Calling set as Auto()
## Important: note that `function_choice_behavior=FunctionChoiceBehavior`

In [14]:
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

execution_settings = AzureChatPromptExecutionSettings()
# Auto(), Required() or NoneInvoke()
execution_settings.function_choice_behavior= FunctionChoiceBehavior.Auto()
execution_settings

AzureChatPromptExecutionSettings(service_id=None, extension_data={}, function_choice_behavior=FunctionChoiceBehavior(enable_kernel_functions=True, maximum_auto_invoke_attempts=5, filters=None, type_=<FunctionChoiceType.AUTO: 'auto'>), ai_model_id=None, frequency_penalty=None, logit_bias=None, max_tokens=None, number_of_responses=None, presence_penalty=None, seed=None, stop=None, stream=False, temperature=None, top_p=None, user=None, store=None, metadata=None, response_format=None, function_call=None, functions=None, messages=None, function_call_behavior=None, parallel_tool_calls=True, tools=None, tool_choice=None, structured_json_response=False, stream_options=None, extra_body=None)

# Invoke the assistant using the new message and the new settings

In [16]:
result = await chat_completion.get_chat_message_contents(
    chat_history=history,
    settings=execution_settings,
    kernel=kernel)

#  Print the results
print(result)

print(f"\nAssistant's response: {result[0].content}")

[ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-An42E48ysWBztsNDCr9UjgxGkrzpK', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The status of the second light, "Porch light", has been toggled. It is now turned on.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1736256946, model='gpt-4o-2024-05-13', object='chat.completion', service_tier=None, system_fingerprint='fp_04751d0b65', usage=CompletionUsage(completion_tokens=24, prompt_tokens=210, total_tokens=234, completion_tokens_details=None, prompt_tokens_details=None), prompt_filter_results=[{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak':

In [17]:
history

ChatHistory(messages=[ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Toggle the status of my second light.', encoding=None)], encoding=None, finish_reason=None), ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-An41vjrkQceqJltt8EdRdaBD7NrCJ', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_0EbYMGlehYPMTRZyfHcBrtyT', function=Function(arguments='{}', name='Lights-get_lights'), type='function')]), content_filter_results={})], created=1736256927, model='gpt-4o-2024-05-13', object='chat.completion', service_tier=None, system_fingerprint='fp_f3927aa00d', usage=CompletionUsage(completion_tokens=13, prompt_tokens=78, tota

In [18]:
for cmc in history.messages: # ChatMessageContent
    if not cmc.inner_content is None:
        for choice in cmc.inner_content.choices:
            for tc in choice.message.tool_calls:
                print (f"Call {tc.function.name}({tc.function.arguments})")

Call Lights-get_lights({})
Call Lights-change_state({"id":1,"is_on":true})


# Additional tests. You may run the next cell multiple times to toggle the first light.

In [ ]:
# Create a blank history of the conversation
from semantic_kernel.contents.chat_history import ChatHistory
history = ChatHistory() # initially blank
history.add_user_message("Toggle the first light and give me the status of all my lights.")

result = await chat_completion.get_chat_message_contents(
    chat_history=history,
    settings=execution_settings,
    kernel=kernel)

#  Print the results
print(f"\nAssistant's response: {result[0].content}")

In [ ]:
history